# 06. 웹 스크래핑 기초 — HTML 에서 표 만들기

`legacy/News screpion.ipynb` 이 하려던 일(검색 결과에서 기사 링크를 모아 CSV 로 저장)을
다시 쓴다. 원본은 지금 실행하면 **결과가 0건**이다. 이유는 코드가 아니라 **사이트가 바뀌어서**다.

```python
soup = soup.find_all("div", attrs={"class": "thumb"})   # 2019년 네이버의 DOM
```

## 이 노트북의 방침
실습은 `data/news_sample.html` — **저장소가 직접 만든 가짜 검색 결과 페이지**로 한다.

* 특정 사이트의 DOM 변경이나 이용약관에 실습이 묶이지 않는다
* 네트워크 없이도 항상 같은 결과가 나온다
* 실제 사이트에 적용할 때 필요한 것은 **선택자뿐**이고, 나머지 구조는 그대로다

## 학습 목표
1. HTML 을 파싱해 원하는 요소 고르기 (`find`, `find_all`, `select`)
2. CSS 선택자 읽는 법
3. 결과를 DataFrame → 중복 제거 → CSV
4. 표는 `pandas.read_html` 한 줄
5. **깨지지 않는 코드**: 요소가 없을 때, 타임아웃, 재시도
6. 스크래핑의 예의와 법 (robots.txt, 요청 간격, 이용약관, 개인정보)

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nbtools").is_dir())
sys.path.insert(0, str(ROOT))
from nbtools import ensure_data  # noqa: E402

ensure_data(quiet=True)
html = (ROOT / "data" / "news_sample.html").read_text(encoding="utf-8")
print(html[:400], "...")

<!doctype html>
<html lang="ko">
<head><meta charset="utf-8"><title>검색 결과 - 업무 자동화</title></head>
<body>
  <header><h1>가상 뉴스 검색</h1></header>
  <main>
    <p class="result-count">총 <strong>18</strong>건</p>
    <ul class="news-list">
    <li class="news-item">
      <a class="news-title" href="/article/1000">RPA 도입 기업 늘어 (1회차)</a>
      <div class="news-meta">
        <span class="press">가상뉴스</span ...


## 1. 파서 고르기

`BeautifulSoup(html, parser)` 의 두 번째 인자를 **생략하지 않는다**. 생략하면 환경에 따라
다른 파서가 선택돼 결과가 달라진다.

| 파서 | 특징 |
|---|---|
| `html.parser` | 표준 라이브러리, 설치 불필요, 조금 느림 |
| `lxml` | 빠름, 깨진 HTML 에 관대 (`pip install lxml`) |
| `html5lib` | 브라우저와 가장 비슷하게 교정, 가장 느림 |

In [2]:
soup = BeautifulSoup(html, "html.parser")
print("제목:", soup.title.string)
print("결과 건수 표기:", soup.select_one("p.result-count strong").get_text())

제목: 검색 결과 - 업무 자동화
결과 건수 표기: 18


## 2. 요소 고르기 — `find_all` 과 `select`

둘 다 쓸 수 있다. **CSS 선택자(`select`)** 쪽이 브라우저 개발자도구에서 복사해 오기 쉬워 실무에서 편하다.

In [3]:
by_find = soup.find_all("li", class_="news-item")
by_select = soup.select("ul.news-list > li.news-item")
print(f"find_all: {len(by_find)}개 / select: {len(by_select)}개")

first = by_select[0]
print("\n첫 항목 HTML:\n", first.prettify()[:300])

find_all: 18개 / select: 18개

첫 항목 HTML:
 <li class="news-item">
 <a class="news-title" href="/article/1000">
  RPA 도입 기업 늘어 (1회차)
 </a>
 <div class="news-meta">
  <span class="press">
   가상뉴스
  </span>
  <span class="date">
   2025-01-01
  </span>
  <span class="views">
   4597
  </span>
 </div>
 <p class="news-summary">
  제조업 중심으로 단순 반복 업


자주 쓰는 선택자 문법:

| 선택자 | 의미 |
|---|---|
| `div.card` | class 가 card 인 div |
| `#main` | id 가 main |
| `a[href]` | href 속성이 있는 a |
| `a[href^="/article"]` | href 가 /article 로 시작 |
| `ul > li` | 직계 자식만 |
| `ul li` | 후손 전부 |
| `li:nth-of-type(2)` | 두 번째 |

In [4]:
link = first.select_one("a.news-title")
print("텍스트  :", link.get_text(strip=True))
print("href    :", link["href"])
print("속성 전체:", link.attrs)
print("언론사  :", first.select_one("span.press").get_text(strip=True))

텍스트  : RPA 도입 기업 늘어 (1회차)
href    : /article/1000
속성 전체: {'class': ['news-title'], 'href': '/article/1000'}
언론사  : 가상뉴스


## 3. 구조화 — 한 항목을 dict 로

**요소가 없을 수 있다**는 전제로 쓴다. `select_one` 은 못 찾으면 `None` 을 주고,
거기에 `.get_text()` 를 부르면 `AttributeError` 로 전체 루프가 죽는다.
실전 HTML 은 항목마다 구조가 조금씩 다르므로 이 방어가 반드시 필요하다.

In [5]:
def text_of(node, selector: str, default: str = "") -> str:
    """선택자에 해당하는 텍스트. 없으면 기본값 — 한 항목이 이상해도 전체가 멈추지 않게."""
    found = node.select_one(selector)
    return found.get_text(strip=True) if found else default


def parse_item(node) -> dict:
    link = node.select_one("a.news-title")
    return {
        "title": link.get_text(strip=True) if link else "",
        "url": link["href"] if link and link.has_attr("href") else "",
        "press": text_of(node, "span.press"),
        "date": text_of(node, "span.date"),
        "views": pd.to_numeric(text_of(node, "span.views"), errors="coerce"),
        "summary": text_of(node, "p.news-summary"),
    }


records = [parse_item(node) for node in soup.select("li.news-item")]
news = pd.DataFrame(records)
print(f"{len(news)}건 수집")
news.head(3)

18건 수집


,title,url,press,date,views,summary
0,RPA 도입 기업 늘어 (1회차),/article/1000,가상뉴스,2025-01-01,4597,제조업 중심으로 단순 반복 업무 자동화가 확산되고 있다.
1,사무 자동화 시장 성장 (2회차),/article/1001,테크타임즈,2025-02-02,5905,국내 사무 자동화 시장이 전년 대비 성장했다는 조사 결과가 나왔다.
2,업무 자동화와 일자리 (3회차),/article/1002,데일리리포트,2025-03-03,3671,자동화가 일자리에 미치는 영향을 두고 논의가 이어지고 있다.


## 4. 정제 — 중복 제거와 절대 URL

원본 노트북도 `drop_duplicates(['url'])` 를 했다. 그 부분은 맞았다.
다만 **상대 URL 을 절대 URL 로 바꾸는 처리**가 없었다 — 나중에 그 링크를 다시 방문할 때 깨진다.

In [6]:
from urllib.parse import urljoin  # noqa: E402

BASE = "https://example.test/search"

news["url"] = news["url"].apply(lambda u: urljoin(BASE, u))
before = len(news)
news = news.drop_duplicates(subset="url", keep="first").reset_index(drop=True)
print(f"중복 제거: {before} → {len(news)}건")
news[["title", "press", "url"]].head()

중복 제거: 18 → 16건


,title,press,url
0,RPA 도입 기업 늘어 (1회차),가상뉴스,https://example.test/article/1000
1,사무 자동화 시장 성장 (2회차),테크타임즈,https://example.test/article/1001
2,업무 자동화와 일자리 (3회차),데일리리포트,https://example.test/article/1002
3,오픈소스 자동화 도구 비교 (4회차),오픈데이터신문,https://example.test/article/1003
4,자동화 도입 실패 사례 (5회차),샘플경제,https://example.test/article/1004


## 5. 저장 — 날짜 폴더에 CSV

원본의 폴더 구조(`scrap/YYYYMMDD/키워드_YYYYMMDD.csv`)는 좋은 아이디어였다. 그대로 살리되
경로 계산만 고친다. 원본은 이렇게 썼다.

```python
base_dir = os.path.dirname(os.path.realpath('__file__')).replace("jupyter_workspace","scrap")
```

`'__file__'` 이 **따옴표 안에 들어간 문자열**이다. 노트북에는 `__file__` 변수가 없어서 넣은 우회인데,
결과적으로 "현재 폴더 + `/__file__`" 의 디렉터리, 즉 현재 폴더를 반환한다. 그리고 폴더명에
`jupyter_workspace` 가 들어 있을 때만 동작한다. 지금은 `pathlib` 로 명시적으로 쓴다.

In [7]:
from datetime import date  # noqa: E402

keyword = "업무 자동화"
out_dir = ROOT / "outputs" / "scrap" / date.today().strftime("%Y%m%d")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f"{keyword.replace(' ', '_')}.csv"
news.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"저장: {out_path.relative_to(ROOT)} ({out_path.stat().st_size:,} bytes)")

저장: outputs/scrap/20260812/업무_자동화.csv (3,055 bytes)


여러 날짜 폴더에 흩어진 CSV 를 다시 하나로 모으는 것도 한 줄이다.

In [8]:
frames = [pd.read_csv(p) for p in sorted((ROOT / "outputs" / "scrap").rglob("*.csv"))]
merged = pd.concat(frames, ignore_index=True).drop_duplicates(subset="url")
print(f"{len(frames)}개 파일 → {len(merged)}건")

1개 파일 → 16건


## 6. 표가 있으면 `read_html` 한 줄

`<table>` 이면 BeautifulSoup 을 쓸 필요가 없다.

In [9]:
tables = pd.read_html(ROOT / "data" / "spec_table.html")
spec = tables[0]
print(f"표 {len(tables)}개 발견")
spec

표 1개 발견


,모델,승차정원,연료,신차가격
0,아반떼,5,가솔린,"1,980만원"
1,쏘렌토,7,디젤,"3,450만원"
2,모델Y,5,전기,"5,690만원"
3,GV80,5,가솔린,"7,120만원"
4,스파크,4,가솔린,"1,180만원"


In [10]:
# 숫자 컬럼 정리 (04 노트북과 같은 요령)
spec["신차가격_만원"] = pd.to_numeric(
    spec["신차가격"].str.replace(",", "", regex=False).str.replace("만원", "", regex=False), errors="coerce"
)
spec[["모델", "연료", "신차가격_만원"]].sort_values("신차가격_만원", ascending=False)

,모델,연료,신차가격_만원
3,GV80,가솔린,7120
2,모델Y,전기,5690
1,쏘렌토,디젤,3450
0,아반떼,가솔린,1980
4,스파크,가솔린,1180


## 7. 실제 사이트를 향할 때 — 견고성

로컬 파일과 달리 네트워크는 **실패가 기본값**이다. 최소한 이 정도는 갖춘다.

In [11]:
import requests  # noqa: E402


def fetch(url: str, *, retries: int = 3, timeout: float = 5.0, delay: float = 1.0) -> str | None:
    """타임아웃·재시도·요청 간격을 갖춘 최소한의 요청 함수.

    * timeout 없는 requests 는 응답이 없으면 **영원히** 기다린다 (기본값이 없다)
    * 429/5xx 는 잠시 뒤 재시도, 4xx 는 즉시 포기 (재시도해도 같은 결과다)
    * User-Agent 에 연락처를 적는 것이 관례 — 문제가 생겼을 때 차단 대신 연락이 온다
    """
    headers = {"User-Agent": "jupyterTest-study/1.0 (+https://github.com/SungmanHan/jupyterTest)"}
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            if response.status_code == 200:
                return response.text
            if 400 <= response.status_code < 500 and response.status_code != 429:
                print(f"  {response.status_code} — 재시도해도 소용없음")
                return None
            print(f"  {response.status_code} — {attempt}/{retries} 재시도")
        except requests.RequestException as exc:
            print(f"  {type(exc).__name__} — {attempt}/{retries} 재시도")
        time.sleep(delay * attempt)      # 지수적으로 간격을 늘린다(백오프)
    return None


print(fetch.__doc__.splitlines()[0])

타임아웃·재시도·요청 간격을 갖춘 최소한의 요청 함수.


## 8. 예의와 법 — 코드보다 중요한 부분

스크래핑은 **기술 문제이기 전에 권한 문제**다. 아래는 최소 기준이다.

| 항목 | 확인 방법 |
|---|---|
| **robots.txt** | `https://사이트/robots.txt` 에서 해당 경로가 `Disallow` 인지 |
| **이용약관** | 자동 수집을 금지하는 조항이 있는지. 금지면 하지 않는다 |
| **공식 API** | API 가 있으면 스크래핑 대신 그쪽을 쓴다 (안정적이고 합법적이다) |
| **요청 간격** | 최소 1초 이상. 동시 요청 남발은 사실상 부하 공격이다 |
| **수집 범위** | 필요한 것만. 사이트 전체 미러링은 하지 않는다 |
| **저작권** | 기사 본문 전체 저장·재배포는 저작권 문제다. 링크·제목·메타데이터 위주로 |
| **개인정보** | 이름·연락처 등은 수집하지 않는다. 공개돼 있다는 것이 수집 근거가 되지 않는다 |

`robots.txt` 는 표준 라이브러리로 확인할 수 있다.

In [12]:
from urllib.robotparser import RobotFileParser  # noqa: E402

def allowed(base: str, path: str, agent: str = "*") -> bool | None:
    """robots.txt 상 접근 허용 여부. 확인 불가면 None (= 모르면 하지 않는다)."""
    parser = RobotFileParser()
    parser.set_url(urljoin(base, "/robots.txt"))
    try:
        parser.read()
    except Exception as exc:                     # 네트워크·파싱 실패
        print("robots.txt 확인 실패:", type(exc).__name__)
        return None
    return parser.can_fetch(agent, urljoin(base, path))

print("사용법: allowed('https://example.com', '/search')  →", allowed.__doc__.splitlines()[0])

사용법: allowed('https://example.com', '/search')  → robots.txt 상 접근 허용 여부. 확인 불가면 None (= 모르면 하지 않는다).


> 연습이 필요하면 **스크래핑 연습용으로 공개된 사이트**를 쓴다
> (`quotes.toscrape.com`, `books.toscrape.com` 등). 실서비스를 연습장으로 삼지 않는다.

## 정리

| 원본 | 지금 |
|---|---|
| 특정 포털 DOM(`div.thumb`) 하드코딩 | 선택자만 갈아끼우면 되는 구조 + 로컬 샘플로 실습 |
| `os.path.realpath('__file__')` | `pathlib.Path` |
| 요소 없으면 AttributeError | `select_one` 결과 None 검사 |
| 타임아웃·재시도 없음 | `timeout` + 백오프 재시도 |
| 상대 URL 그대로 저장 | `urljoin` 으로 절대화 |
| robots.txt·간격 고려 없음 | 확인 절차를 코드로 |

다음: **07. 동적 페이지와 Selenium** — 원본이 Selenium 을 쓴 이유와, 2019년 문법이 왜 지금 안 되는지.